# F2.4 · Transformación de variables

Se asigna a cada variable su tipo, se crea la variable `modalidad` y se deja una fila por orden.

In [1]:
# Configuración común: rutas del proyecto y módulos del pipeline.
import pandas as pd

from pipeline import chilecompra as cc
from pipeline import config, ordenes

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.1f}".format)
print("Raíz del proyecto:", config.PROJECT_ROOT.name)

Raíz del proyecto: 002_sumativa-1-avance-proyecto-f1-y-f2-40


In [2]:
# Etapas anteriores, reutilizadas desde src/pipeline/ordenes.py.
raw = ordenes.load_orders(config.RAW_DATA_DIR)
clean = ordenes.normalize_missing(raw)

## 1. Conversión de tipos (casting)

- **Montos:** de texto con coma decimal a número. Se usa `MontoTotalOC_PesosChilenos`, que ya trae en pesos las pocas órdenes emitidas en UF, UTM o dólares.
- **Fechas:** de texto a fecha.
- **Sí/No:** a booleano que admite NA, para no convertir un dato ausente en un «No».

In [3]:
typed = ordenes.cast_types(clean)
columns = ["MontoTotalOC_PesosChilenos", "totalLineaNeto", "FechaEnvio", "EsTratoDirecto", "EsCompraAgil"]
pd.DataFrame({"antes": clean[columns].dtypes.astype(str), "después": typed[columns].dtypes.astype(str), "NA después": typed[columns].isna().sum()})

,antes,después,NA después
MontoTotalOC_PesosChilenos,str,float64,0
totalLineaNeto,str,float64,0
FechaEnvio,str,datetime64[us],0
EsTratoDirecto,str,boolean,5270
EsCompraAgil,str,boolean,0


In [4]:
# La conversión no debe crear NA nuevos en montos ni en la fecha de envío.
assert typed["MontoTotalOC_PesosChilenos"].isna().sum() == clean["MontoTotalOC_PesosChilenos"].isna().sum()
assert typed["FechaEnvio"].notna().all()
print("Ejemplo:", clean["MontoTotalOC_PesosChilenos"].iloc[0], "->", typed["MontoTotalOC_PesosChilenos"].iloc[0])

Ejemplo: 3401030,71 -> 3401030.71


## 2. Variable derivada: modalidad de compra

`classify_modality` etiqueta cada orden con una modalidad **comparable entre años**. La compra ágil se evalúa primero: así una compra ágil de 2024 (que además venía marcada como trato directo) recibe la misma etiqueta que una de 2025.

| Orden de evaluación | Condición | Modalidad |
|---|---|---|
| 1 | `EsCompraAgil` = Sí | Compra ágil |
| 2 | `EsTratoDirecto` = Sí | Trato directo |
| 3 | `Tipo` = CM | Convenio marco |
| 4 | Tiene `CodigoLicitacion` | Licitación |
| 5 | Ninguna de las anteriores | Otra |

**Enfoque descartado:** usar `EsTratoDirecto` tal como viene. Mostraría que el trato directo cayó a la mitad en 2025, cuando en realidad cambió la forma de registrarlo.

In [5]:
labeled = ordenes.classify_modality(typed)
pd.crosstab(labeled["periodo"], labeled["modalidad"])

modalidad,Compra ágil,Convenio marco,Licitación,Otra,Trato directo
periodo,,,,,
2024-09,4013,1818,3974,207,2013
2025-09,5306,2044,4515,240,1750


## 3. Agregación: una fila por orden

El monto total de la orden se repite en cada una de sus líneas. Sumarlo sobre las líneas lo cuenta varias veces.

In [6]:
orders = ordenes.aggregate_orders(labeled)
inflated = labeled["MontoTotalOC_PesosChilenos"].sum() / 1e6
correct = orders["MontoTotalOC_PesosChilenos"].sum() / 1e6
print(f"Suma sobre líneas (incorrecta): ${inflated:,.0f} millones")
print(f"Suma sobre órdenes (correcta):  ${correct:,.0f} millones")
print(f"El error habría sido de {inflated / correct:.1f} veces")

Suma sobre líneas (incorrecta): $191,009 millones
Suma sobre órdenes (correcta):  $49,163 millones
El error habría sido de 3.9 veces


**Sobre normalización y escalamiento:** no se aplican en esta fase. Los indicadores del estudio son proporciones y sumas de montos en pesos, que deben conservar su unidad. El escalamiento se evaluará en la Fase 3 si algún algoritmo lo requiere.

## 4. Dataset procesado y diccionario final

In [7]:
config.PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
output = config.PROCESSED_DATA_DIR / "ordenes.csv"
orders.to_csv(output, index=False)
print("Guardado:", output.relative_to(config.PROJECT_ROOT), f"({len(orders):,} órdenes, {orders.shape[1]} variables)")

pd.DataFrame({"tipo": orders.dtypes.astype(str), "NA": orders.isna().sum(), "valores distintos": orders.nunique(), "ejemplo": orders.iloc[0].astype(str)})

Guardado: data/processed/ordenes.csv (10,285 órdenes, 13 variables)


,tipo,NA,valores distintos,ejemplo
periodo,str,0,2,2024-09
Codigo,str,0,10285,2200-524-SE23
FechaEnvio,datetime64[us],0,50,2024-09-27 00:00:00
Estado,str,0,5,Enviada a proveedor
modalidad,str,0,5,Trato directo
TipoMonedaOC,str,0,4,CLP
MontoTotalOC_PesosChilenos,float64,0,7211,3401030.71
CodigoUnidadCompra,str,0,57,3195
UnidadCompra,str,0,54,Servicio de Salud Aconcagua
OrganismoPublico,str,0,29,SERVICIO DE SALUD ACONCAGUA
